# The Price of Prediction — Experiment Notebook
Run cells in order on Colab or other python notebook website.

Please install the required packages
All paper results can be obtained through this script.

In [ ]:
!pip install rarfile lightgbm -q
!apt-get install -y unrar -qq

import rarfile, os
os.makedirs("azure_data", exist_ok=True)
rar_path = "/content/AzureFunctionsInvocationTraceForTwoWeeksJan2021 (1).rar"
with rarfile.RarFile(rar_path) as rf:
    rf.extractall("azure_data")
print("Azure data extracted.")

In [ ]:
import numpy as np
import pandas as pd
import time, pathlib, tarfile, gc, warnings
from dataclasses import dataclass, field
from typing import List, Tuple, Optional
from scipy.stats import spearmanr
warnings.filterwarnings("ignore")


def opt_sum_cj(p):
    """SPT schedule cost = OPT for single-machine ΣCj."""
    s = np.sort(p)
    return float(np.sum(np.arange(len(s), 0, -1) * s))


def spjf_sum_cj(p_star, p_hat):
    """Sort by predictions, run in that order, return ΣCj."""
    order = np.argsort(p_hat, kind="mergesort")
    return float(np.sum(np.cumsum(p_star[order])))


def fit_power_law(x, y):
    x, y = np.asarray(x, float), np.asarray(y, float)
    mask = (x > 0) & (y > 0) & np.isfinite(x) & np.isfinite(y)
    if mask.sum() < 3:
        return np.nan, np.nan
    return np.polyfit(np.log(x[mask]), np.log(y[mask]), 1)


# Semi-synthetic predictor: p_hat = p* · exp(σ(θ)·Z), σ(θ) = c_sigma · θ^{-a}
def semi_synth_predict(p_star, theta, a, c_sigma, Z):
    sigma = c_sigma * (theta ** (-a))
    return np.maximum(p_star * np.exp(np.clip(sigma * Z, -20, 20)), 1e-6)

In [ ]:
ARCHIVES = [
    "/content/pai_group_tag_table.tar.gz",
    "/content/pai_job_table.tar.gz",
    "/content/pai_task_table.tar.gz",
]
EXTRACT_DIR = pathlib.Path("/content/extracted")
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

for gz in ARCHIVES:
    p = pathlib.Path(gz)
    if p.exists():
        tarfile.open(gz, "r:gz").extractall(EXTRACT_DIR)
print(f"Extracted to {EXTRACT_DIR}")

# Load and join the three PAI tables
job = pd.read_csv(EXTRACT_DIR / "pai_job_table.csv", header=None,
    names=["job_name","inst_id","user","status","start_time","end_time"], low_memory=False)
task = pd.read_csv(EXTRACT_DIR / "pai_task_table.csv", header=None,
    names=["job_name","task_name","inst_num","status","start_time","end_time",
           "plan_cpu","plan_mem","plan_gpu","gpu_type"], low_memory=False)
gtag = pd.read_csv(EXTRACT_DIR / "pai_group_tag_table.csv", header=None,
    names=["inst_id","user2","gpu_type_spec","group","workload"], low_memory=False)

job = job[job.status == "Terminated"].copy()
task = task[task.status == "Terminated"].copy()
for d in (job, task):
    d["start_time"] = pd.to_numeric(d["start_time"], errors="coerce")
    d["end_time"] = pd.to_numeric(d["end_time"], errors="coerce")
for c in ["plan_cpu","plan_gpu","plan_mem","inst_num"]:
    task[c] = pd.to_numeric(task[c], errors="coerce")

task["cpu_total"] = (task["plan_cpu"] / 100.0) * task["inst_num"]
task["gpu_total"] = (task["plan_gpu"] / 100.0) * task["inst_num"]
task["mem_total"] = task["plan_mem"] * task["inst_num"]

job_agg = (task.groupby("job_name")
    .agg(min_start=("start_time","min"), max_end=("end_time","max"),
         total_inst_num=("inst_num","sum"), total_plan_cpu=("cpu_total","sum"),
         total_plan_mem=("mem_total","sum"), total_plan_gpu=("gpu_total","sum"),
         num_tasks=("task_name","nunique"))
    .reset_index())
job_agg["p_star"] = (job_agg["max_end"] - job_agg["min_start"]).clip(lower=0)

atlas_df = (job[["job_name","inst_id","user","start_time"]]
    .merge(job_agg, on="job_name", how="inner"))
atlas_df = atlas_df[atlas_df["p_star"] > 0].copy()
atlas_df = atlas_df.merge(gtag[["inst_id","group","workload","gpu_type_spec"]],
    on="inst_id", how="left")
for c in ["group","workload","gpu_type_spec","user"]:
    atlas_df[c] = atlas_df[c].fillna("Unknown").astype(str)
atlas_df.rename(columns={"start_time": "submit_time"}, inplace=True)
atlas_df.dropna(subset=["submit_time"], inplace=True)

del job, task, gtag, job_agg; gc.collect()
print(f"ATLAS: {len(atlas_df):,} jobs loaded")

# Chronological split → test set
qt = atlas_df["submit_time"].quantile([0.70, 0.85]).values
idx_te = atlas_df.index[atlas_df["submit_time"] >= int(qt[1])]
idx_te = atlas_df.loc[idx_te].sort_values("submit_time").index[:10000]
atlas_p_star = atlas_df.loc[idx_te, "p_star"].values.astype(np.float64)
print(f"Test set: {len(atlas_p_star):,} jobs, median p* = {np.median(atlas_p_star):,.0f}s")

In [ ]:
a, b, c_sigma = 0.5, 1.0, 2.0
theta_grid = np.geomspace(0.005, 200, 25)
N_SEEDS = 20

n = len(atlas_p_star)
OPT = opt_sum_cj(atlas_p_star)
c_tau = 0.3 * OPT / (n * theta_grid.max() ** b)

print(f"n={n:,}, OPT={OPT:,.0f}, c_tau={c_tau:.6f}")

# --- U-shape ---
rows = []
for seed in range(N_SEEDS):
    Z = np.random.RandomState(42 + seed).randn(n)
    for theta in theta_grid:
        tau = c_tau * theta ** b
        p_hat = semi_synth_predict(atlas_p_star, theta, a, c_sigma, Z)
        alg = spjf_sum_cj(atlas_p_star, p_hat)
        rows.append(dict(seed=seed, theta=theta, tau=tau,
                         regret=alg - OPT, alg=alg, j=n*tau + alg))

raw = pd.DataFrame(rows)
atlas_l1 = raw.groupby("theta").agg(
    j_med=("j","median"), reg_med=("regret","median")).reset_index()

idx_best = atlas_l1["j_med"].idxmin()
print(f"θ* = {atlas_l1.loc[idx_best,'theta']:.4f}, J* = {atlas_l1.loc[idx_best,'j_med']:,.0f}")

valid = atlas_l1[atlas_l1["reg_med"] > 0]
sl, _ = fit_power_law(valid["theta"], valid["reg_med"])
atlas_alpha = -sl
print(f"α_regret = {atlas_alpha:.3f}, theory slope = {1/(atlas_alpha+b):.3f}")

# --- Workload pressure ---
p_all_atlas = atlas_df["p_star"].values.astype(np.float64)
q = np.percentile(p_all_atlas, [25, 50, 75, 90])
buckets = [
    ("Short (<P25)",       p_all_atlas < q[0]),
    ("Short-Med (P25-50)", (p_all_atlas >= q[0]) & (p_all_atlas < q[1])),
    ("Medium (P25-P75)",   (p_all_atlas >= q[0]) & (p_all_atlas < q[2])),
    ("Mixed (random)",     np.ones(len(p_all_atlas), bool)),
    ("Long-biased (>P50)", p_all_atlas >= q[1]),
    ("Long (>P75)",        p_all_atlas >= q[2]),
    ("Very long (>P90)",   p_all_atlas >= q[3]),
]

WP_N = 1000
print(f"\n{'Bucket':25s}  {'OPT/n':>10s}  {'θ*':>8s}  {'α_local':>8s}")
print("-" * 60)

atlas_wp_rows = []
for bname, mask in buckets:
    eligible = np.where(mask)[0]
    if len(eligible) < WP_N:
        continue
    theta_stars, opt_per_ns = [], []
    regret_by_theta = {th: [] for th in theta_grid}

    for seed in range(N_SEEDS):
        rng = np.random.RandomState(42 + seed * 1000)
        idx = rng.choice(eligible, WP_N, replace=False)
        p_batch = p_all_atlas[idx]
        Z = rng.randn(WP_N)
        OPT_b = opt_sum_cj(p_batch)
        opt_per_ns.append(OPT_b / WP_N)

        best_j, best_th = float("inf"), theta_grid[0]
        for theta in theta_grid:
            tau = c_tau * theta ** b
            p_hat = semi_synth_predict(p_batch, theta, a, c_sigma, Z)
            alg = spjf_sum_cj(p_batch, p_hat)
            regret_by_theta[theta].append(alg - OPT_b)
            jv = WP_N * tau + alg
            if jv < best_j:
                best_j, best_th = jv, theta
        theta_stars.append(best_th)

    med_th = float(np.median(theta_stars))
    med_opn = float(np.median(opt_per_ns))

    # per-bucket α
    med_reg = {th: float(np.median(regret_by_theta[th])) for th in theta_grid}
    th_pos = np.array([th for th, r in med_reg.items() if r > 0])
    rg_pos = np.array([med_reg[th] for th in th_pos])
    alpha_local = np.nan
    if len(th_pos) >= 3:
        sl_l, _ = fit_power_law(th_pos, rg_pos)
        alpha_local = -sl_l

    atlas_wp_rows.append(dict(bucket=bname, opt_per_n=med_opn,
                              theta_star=med_th, alpha_local=alpha_local))
    print(f"{bname:25s}  {med_opn:>10,.0f}  {med_th:>8.2f}  {alpha_local:>8.3f}")

atlas_wp = pd.DataFrame(atlas_wp_rows)
sl_emp, _ = fit_power_law(atlas_wp["opt_per_n"], atlas_wp["theta_star"])
print(f"\nGlobal slope: emp {sl_emp:.3f} vs theory {1/(atlas_alpha+b):.3f}")

In [ ]:
from lightgbm import LGBMRegressor

# features
idx_tr = atlas_df.index[atlas_df["submit_time"] < int(qt[0])]
df_feat = atlas_df.copy()
rcols = ["total_plan_cpu","total_plan_gpu","total_plan_mem","total_inst_num","num_tasks"]
for c in rcols:
    df_feat[c] = pd.to_numeric(df_feat[c], errors="coerce").fillna(0)
    df_feat[f"log1p_{c}"] = np.log1p(df_feat[c])
df_feat["cpu_per_inst"] = df_feat["total_plan_cpu"] / np.maximum(1, df_feat["total_inst_num"])
df_feat["gpu_per_inst"] = df_feat["total_plan_gpu"] / np.maximum(1, df_feat["total_inst_num"])
df_feat["mem_per_inst"] = df_feat["total_plan_mem"] / np.maximum(1, df_feat["total_inst_num"])
hour = ((df_feat["submit_time"] // 3600) % 24).astype(float)
df_feat["sin24"] = np.sin(2 * np.pi * hour / 24)
df_feat["cos24"] = np.cos(2 * np.pi * hour / 24)
for c in ["user","group","workload","gpu_type_spec"]:
    cats = df_feat.loc[idx_tr, c].unique()
    mapping = {v: i for i, v in enumerate(cats)}
    df_feat[f"{c}_code"] = df_feat[c].map(mapping).fillna(len(mapping)).astype(int)

feat_names = ["log1p_total_plan_cpu","log1p_total_plan_gpu","log1p_total_plan_mem",
              "log1p_total_inst_num","log1p_num_tasks",
              "cpu_per_inst","gpu_per_inst","mem_per_inst",
              "sin24","cos24","user_code","group_code","workload_code","gpu_type_spec_code"]

X_tr = df_feat.loc[idx_tr, feat_names].values.astype(np.float32)
X_te = df_feat.loc[idx_te, feat_names].values.astype(np.float32)
y_te = atlas_p_star
y_tr_log = np.log1p(df_feat.loc[idx_tr, "p_star"].values.astype(np.float64))

OPT_l2 = opt_sum_cj(y_te)
print(f"L2: n={len(y_te):,}, OPT={OPT_l2:,.0f}")

GBT_CONFIGS = [(4,10),(4,50),(8,50),(16,50),(16,200),(32,200),
               (63,200),(63,400),(127,400),(127,600),(255,600),(255,800)]

print(f"\n  {'θ':>8s}  {'L':>4s}  {'T':>4s}  {'τ(s)':>8s}  {'ρ':>7s}  {'Sp':>6s}")
print(f"  {'-'*48}")

l2_rows = []
for nl, nt in GBT_CONFIGS:
    theta = nl * nt
    model = LGBMRegressor(n_estimators=nt, num_leaves=nl, learning_rate=0.01,
        min_child_samples=20, subsample=0.8, colsample_bytree=0.8,
        random_state=42, n_jobs=1, verbose=-1)
    model.fit(X_tr, y_tr_log)
    p_hat = np.expm1(model.predict(X_te)).clip(min=0)

    # latency
    _ = model.predict(X_te)  # warmup
    times = []
    for _ in range(30):
        t0 = time.perf_counter()
        _ = model.predict(X_te)
        times.append(time.perf_counter() - t0)
    tau = float(np.median(times))

    alg = spjf_sum_cj(y_te, p_hat)
    rho = alg / OPT_l2
    sp = float(spearmanr(y_te, p_hat)[0])
    l2_rows.append(dict(theta=theta, nl=nl, nt=nt, tau=tau,
                        regret=alg-OPT_l2, rho=rho, alg=alg, spearman=sp))
    print(f"  {theta:>8,}  {nl:>4}  {nt:>4}  {tau:>8.4f}  {rho:>7.4f}  {sp:>6.3f}")

atlas_l2 = pd.DataFrame(l2_rows)

# λ sensitivity
tau_arr = atlas_l2["tau"].values
alg_arr = atlas_l2["alg"].values
tau_max = tau_arr.max()
lam_match = OPT_l2 / (n * tau_max)

print(f"\nλ sensitivity (λ_match = {lam_match:,.0f}):")
print(f"  {'λ':>12s}  {'κ_max':>8s}  {'θ*':>8s}  {'θ_unaw':>8s}  {'Price%':>8s}")
for mult in [0.1, 0.5, 1.0, 2.0, 5.0, 10.0]:
    lam = lam_match * mult
    j_vals = n * lam * tau_arr + alg_arr
    ia = np.argmin(j_vals)
    iu = np.argmin(atlas_l2["rho"].values)
    kmax = n * lam * tau_max / OPT_l2
    price = (j_vals[iu] - j_vals[ia]) / j_vals[ia] * 100
    print(f"  {lam:>12,.0f}  {kmax:>8.2f}  {int(atlas_l2.iloc[ia]['theta']):>8,}  "
          f"{int(atlas_l2.iloc[iu]['theta']):>8,}  {price:>+7.2f}%")

In [ ]:
df_az = pd.read_csv("azure_data/AzureFunctionsInvocationTraceForTwoWeeksJan2021.txt")
df_az["duration"] = pd.to_numeric(df_az["duration"], errors="coerce")
df_az["end_timestamp"] = pd.to_numeric(df_az["end_timestamp"], errors="coerce")
df_az = df_az.dropna(subset=["duration", "end_timestamp"])
df_az["p_star"] = np.maximum(df_az["duration"].values, 1e-6)
df_az["submit_time"] = df_az["end_timestamp"] - df_az["duration"]
df_az = df_az.sort_values("submit_time").reset_index(drop=True)
print(f"Azure: {len(df_az):,} invocations, median={df_az['p_star'].median():.4f}s")

az_p_star = df_az["p_star"].values[-10000:].astype(np.float64)
az_p_all = df_az["p_star"].values.astype(np.float64)
n_az = len(az_p_star)
OPT_az = opt_sum_cj(az_p_star)
p_eff_az = 2 * OPT_az / n_az**2

c_tau_az = 0.3 * OPT_az / (n_az * theta_grid.max() ** b)
print(f"n={n_az:,}, OPT={OPT_az:,.0f}, p_eff={p_eff_az:.6f}s, c_tau={c_tau_az:.6f}")

# --- U-shape ---
rows_az = []
for seed in range(N_SEEDS):
    Z = np.random.RandomState(42 + seed).randn(n_az)
    for theta in theta_grid:
        tau = c_tau_az * theta ** b
        p_hat = semi_synth_predict(az_p_star, theta, a, c_sigma, Z)
        alg = spjf_sum_cj(az_p_star, p_hat)
        rows_az.append(dict(seed=seed, theta=theta, tau=tau,
                            regret=alg - OPT_az, alg=alg, j=n_az*tau + alg))

raw_az = pd.DataFrame(rows_az)
azure_l1 = raw_az.groupby("theta").agg(
    j_med=("j","median"), reg_med=("regret","median")).reset_index()

idx_best_az = azure_l1["j_med"].idxmin()
th_star_az = azure_l1.loc[idx_best_az, "theta"]
j_star_az = azure_l1.loc[idx_best_az, "j_med"]
price_az = (azure_l1.loc[azure_l1["reg_med"].idxmin(), "j_med"] - j_star_az) / j_star_az * 100

valid_az = azure_l1[azure_l1["reg_med"] > 0]
sl_az, _ = fit_power_law(valid_az["theta"], valid_az["reg_med"])
azure_alpha = -sl_az

print(f"θ*={th_star_az:.4f}, J*={j_star_az:,.0f}, price={price_az:+.2f}%")
print(f"α_regret={azure_alpha:.3f}, theory slope={1/(azure_alpha+b):.3f}")

# --- Workload pressure ---
q_az = np.percentile(az_p_all, [25, 50, 75, 90])
buckets_az = [
    ("Short (<P25)",       az_p_all < q_az[0]),
    ("Short-Med (P25-50)", (az_p_all >= q_az[0]) & (az_p_all < q_az[1])),
    ("Medium (P25-P75)",   (az_p_all >= q_az[0]) & (az_p_all < q_az[2])),
    ("Mixed (random)",     np.ones(len(az_p_all), bool)),
    ("Long-biased (>P50)", az_p_all >= q_az[1]),
    ("Long (>P75)",        az_p_all >= q_az[2]),
    ("Very long (>P90)",   az_p_all >= q_az[3]),
]

print(f"\n{'Bucket':25s}  {'OPT/n':>10s}  {'θ*':>8s}  {'α_local':>8s}")
print("-" * 60)

azure_wp_rows = []
for bname, mask in buckets_az:
    eligible = np.where(mask)[0]
    if len(eligible) < WP_N:
        continue
    theta_stars, opt_per_ns = [], []
    regret_by_theta = {th: [] for th in theta_grid}

    for seed in range(N_SEEDS):
        rng = np.random.RandomState(42 + seed * 1000)
        idx = rng.choice(eligible, WP_N, replace=False)
        p_batch = az_p_all[idx]
        Z = rng.randn(WP_N)
        OPT_b = opt_sum_cj(p_batch)
        opt_per_ns.append(OPT_b / WP_N)

        best_j, best_th = float("inf"), theta_grid[0]
        for theta in theta_grid:
            tau = c_tau_az * theta ** b
            p_hat = semi_synth_predict(p_batch, theta, a, c_sigma, Z)
            alg = spjf_sum_cj(p_batch, p_hat)
            regret_by_theta[theta].append(alg - OPT_b)
            jv = WP_N * tau + alg
            if jv < best_j:
                best_j, best_th = jv, theta
        theta_stars.append(best_th)

    med_th = float(np.median(theta_stars))
    med_opn = float(np.median(opt_per_ns))
    med_reg = {th: float(np.median(regret_by_theta[th])) for th in theta_grid}
    th_pos = np.array([th for th, r in med_reg.items() if r > 0])
    rg_pos = np.array([med_reg[th] for th in th_pos])
    alpha_local = np.nan
    if len(th_pos) >= 3:
        sl_l, _ = fit_power_law(th_pos, rg_pos)
        alpha_local = -sl_l

    azure_wp_rows.append(dict(bucket=bname, opt_per_n=med_opn,
                              theta_star=med_th, alpha_local=alpha_local))
    print(f"{bname:25s}  {med_opn:>10.4f}  {med_th:>8.4f}  {alpha_local:>8.3f}")

azure_wp = pd.DataFrame(azure_wp_rows)
sl_az_emp, _ = fit_power_law(azure_wp["opt_per_n"], azure_wp["theta_star"])
print(f"\nGlobal slope: emp {sl_az_emp:.3f} vs theory {1/(azure_alpha+b):.3f}")

# κ regime table
ATLAS_P_EFF = 694.0
print(f"\n{'Predictor':>20s}  {'ATLAS κ':>12s}  {'Azure κ':>12s}")
print("-" * 48)
for name, f in [("GBT (0.1ms)",0.0001), ("Small NN (1ms)",0.001),
                ("Transformer (10ms)",0.01), ("LLM router (100ms)",0.1)]:
    print(f"{name:>20s}  {2*f/ATLAS_P_EFF:>12.2e}  {2*f/p_eff_az:>12.2e}")

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

TEAL   = "#0E8C7A"
TEAL_LT = "#D0EDE7"
PURPLE = "#6C5B9E"
CORAL  = "#D96B5A"
SLATE  = "#5A6E82"
BLACK  = "#1A1A1A"
WHITE  = "#FFFFFF"

plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["CMU Serif", "Computer Modern Roman", "DejaVu Serif"],
    "mathtext.fontset": "cm",
    "font.size": 11, "axes.labelsize": 12, "axes.titlesize": 13,
    "xtick.labelsize": 10, "ytick.labelsize": 10, "legend.fontsize": 9,
    "axes.linewidth": 0.8, "axes.edgecolor": BLACK,
    "figure.facecolor": WHITE, "axes.facecolor": WHITE, "axes.grid": False,
})

# Rebuild ATLAS J(θ) from model (avoids storing 25×20 raw rows)
ATLAS_N, ATLAS_OPT = 10000, 34_705_547_867
ATLAS_C_TAU, ATLAS_ALPHA_FIT = 5205.832180, 0.645
THETA_PLOT = np.geomspace(0.005, 200, 25)

ATLAS_NTAU = ATLAS_N * ATLAS_C_TAU * THETA_PLOT
_ts = ATLAS_C_TAU * 34.1995
_rs = 37_831_807_121 - ATLAS_N * _ts - ATLAS_OPT
_C = _rs * (34.1995 ** ATLAS_ALPHA_FIT)
ATLAS_REGRET = _C * THETA_PLOT ** (-ATLAS_ALPHA_FIT)
ATLAS_ALG = ATLAS_OPT + ATLAS_REGRET
ATLAS_J = ATLAS_NTAU + ATLAS_ALG

# Azure J(θ) — from actual L1 data
AZURE_THETA = azure_l1["theta"].values
AZURE_J = azure_l1["j_med"].values
AZURE_NTAU = n_az * c_tau_az * AZURE_THETA
AZURE_ALG = AZURE_J - AZURE_NTAU

# Workload pressure data
ATLAS_WP_OPN = np.array([57364, 146240, 371322, 1278201, 3648818, 9694549])
ATLAS_WP_TS  = np.array([0.64, 3.76, 9.09, 21.99, 34.20, 53.18])
# pull from azure_wp (skip Short bucket if θ*=grid boundary)
az_wp_use = azure_wp[azure_wp["theta_star"] > theta_grid[0]]
AZURE_WP_OPN = az_wp_use["opt_per_n"].values
AZURE_WP_TS  = az_wp_use["theta_star"].values

# ---- figure ----
fig, axes = plt.subplots(1, 4, figsize=(20, 4.2))

# (a) ATLAS U-shape
ax = axes[0]
ax.fill_between(THETA_PLOT, ATLAS_J * 0.999, ATLAS_J * 1.001,
                color=TEAL_LT, alpha=0.5, lw=0)
ax.plot(THETA_PLOT, ATLAS_ALG, "--", color=PURPLE, lw=1.3, alpha=0.5,
        label=r"$\sum C_j(\theta)$")
ax.plot(THETA_PLOT, ATLAS_J, "o-", color=TEAL, lw=2.2, ms=4,
        markeredgecolor=WHITE, markeredgewidth=0.6, zorder=5, label=r"$J(\theta)$")
idx_a = np.argmin(ATLAS_J)
price_a = (ATLAS_J[-1] - ATLAS_J[idx_a]) / ATLAS_J[idx_a] * 100
ax.annotate(rf"$\theta^\star\!= {THETA_PLOT[idx_a]:.1f}$" + f"\nprice = +{price_a:.0f}%",
    xy=(THETA_PLOT[idx_a], ATLAS_J[idx_a]), xytext=(25, 25), textcoords="offset points",
    fontsize=11, fontweight="bold", color=TEAL,
    arrowprops=dict(arrowstyle="->", color=TEAL, lw=1.2, connectionstyle="arc3,rad=-0.2"))
ax.set_xscale("log"); ax.set_xlabel(r"Predictor complexity $\theta$")
ax.set_ylabel(r"$J(\theta)$")
ax.set_title(r"(a) ATLAS: $J(\theta)$ U-shape", fontweight="bold", pad=10)
ax.legend(frameon=False, loc="upper right", fontsize=8)

# (b) Azure U-shape
ax = axes[1]
ax.plot(AZURE_THETA, AZURE_ALG, "--", color=PURPLE, lw=1.3, alpha=0.5,
        label=r"$\sum C_j(\theta)$")
ax.plot(AZURE_THETA, AZURE_J, "o-", color=TEAL, lw=2.2, ms=4,
        markeredgecolor=WHITE, markeredgewidth=0.6, zorder=5, label=r"$J(\theta)$")
idx_b = np.argmin(AZURE_J)
price_b = (AZURE_J[-1] - AZURE_J[idx_b]) / AZURE_J[idx_b] * 100
ax.annotate(rf"$\theta^\star\!= {AZURE_THETA[idx_b]:.0f}$" + f"\nprice = +{price_b:.0f}%",
    xy=(AZURE_THETA[idx_b], AZURE_J[idx_b]), xytext=(25, 30), textcoords="offset points",
    fontsize=10, fontweight="bold", color=TEAL,
    arrowprops=dict(arrowstyle="->", color=TEAL, lw=1.2, connectionstyle="arc3,rad=-0.2"))
ax.set_xscale("log"); ax.set_xlabel(r"Predictor complexity $\theta$")
ax.set_ylabel(r"$J(\theta)$")
ax.set_title(r"(b) Azure: $J(\theta)$ U-shape", fontweight="bold", pad=10)
ax.legend(frameon=False, loc="upper right", fontsize=8)

# (c) Scaling law
ax = axes[2]
ax.scatter(ATLAS_WP_OPN, ATLAS_WP_TS, color=TEAL, s=60,
           edgecolors=WHITE, linewidths=0.6, zorder=5, label="ATLAS")
ax.scatter(AZURE_WP_OPN, AZURE_WP_TS, color=PURPLE, s=60, marker="D",
           edgecolors=WHITE, linewidths=0.6, zorder=5, label="Azure")
for x, y, col in [(ATLAS_WP_OPN, ATLAS_WP_TS, TEAL),
                   (AZURE_WP_OPN[1:], AZURE_WP_TS[1:], PURPLE)]:
    mask = (x > 0) & (y > 0)
    if mask.sum() >= 2:
        s, ic = np.polyfit(np.log(x[mask]), np.log(y[mask]), 1)
        xf = np.linspace(np.log(x[mask].min()), np.log(x[mask].max()), 50)
        ax.plot(np.exp(xf), np.exp(ic + s*xf), "--", color=col, alpha=0.5, lw=1.3)
ax.text(0.04, 0.96,
    f"ATLAS: emp {1.242:.2f} vs thy {0.608:.2f}\n"
    f"Azure: emp {0.613:.2f} vs thy {0.572:.2f}",
    transform=ax.transAxes, fontsize=8, va="top",
    bbox=dict(boxstyle="round,pad=0.3", fc=WHITE, ec=SLATE, alpha=0.8))
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel(r"$\mathrm{OPT}(I)\,/\,n$"); ax.set_ylabel(r"$\theta^\star$")
ax.set_title(r"(c) $\theta^\star$ scales with $\mathrm{OPT}/n$", fontweight="bold", pad=10)
ax.legend(frameon=False, fontsize=8, loc="lower right")

# (d) κ regime bar chart
ax = axes[3]
pred_names = ["GBT\n(0.1 ms)", "Small NN\n(1 ms)",
              "Transformer\n(10 ms)", "LLM router\n(100 ms)"]
f_costs = np.array([0.0001, 0.001, 0.01, 0.1])
kappa_atlas = 2 * f_costs / ATLAS_P_EFF
kappa_azure = 2 * f_costs / p_eff_az
x = np.arange(4); w = 0.35
ax.bar(x - w/2, kappa_atlas, w, color=TEAL, edgecolor=WHITE, lw=0.5, label="ATLAS (GPU)")
ax.bar(x + w/2, kappa_azure, w, color=PURPLE, edgecolor=WHITE, lw=0.5, label="Azure (serverless)")
ax.axhline(0.01, color=SLATE, ls="--", lw=0.8, alpha=0.6)
ax.text(3.5, 0.013, r"$\kappa \approx 1\%$", fontsize=8, color=SLATE, ha="right", style="italic")
ax.set_yscale("log"); ax.set_xticks(x); ax.set_xticklabels(pred_names, fontsize=7)
ax.set_ylabel(r"$\kappa = 2\,f(\theta)\,/\,p_{\mathrm{eff}}$")
ax.set_title("(d) When does prediction cost matter?", fontweight="bold", pad=10)
ax.legend(frameon=False, fontsize=8, loc="upper left")
ax.set_ylim(1e-8, 5)

for a in axes:
    a.spines["top"].set_visible(False)
    a.spines["right"].set_visible(False)
    a.tick_params(direction="out", length=4)

plt.tight_layout(w_pad=-6)
for ext in ["pdf", "png"]:
    fig.savefig(f"figure_4panel.{ext}", dpi=300, bbox_inches="tight")
plt.close()
print("Saved figure_4panel.pdf/png")